# MONITORAMENTO PIPELINE CAMADA PRATA_TS_ALUNO

**Pipeline:** Análise da Alfabetização no Brasil


**Objetivo:**

 - Monitorar falhas de leitura

 - Medir latência

 - Monitorar volume processado

 - Validar qualidade dos dados

 - Gerar alertas

 - Verificação de duplicidade

 - Detecção de valores ausentes
 
 - Validação de chaves de relacionamento

## 1. CONFIGURAÇÃO DE LOG

Configura o sistema de logs do pipeline, registrando eventos, métricas e erros em um arquivo para fins de monitoramento

In [1]:
import logging

logging.basicConfig(
filename="monitoramento_prata.log",
level=logging.INFO,
format="%(asctime)s - %(levelname)s - %(message)s"
)

StatementMeta(sparkpool, 31, 2, Finished, Available, Finished, False)

## 2. Configuração de Acesso

Conexão ao ADLS Gen2 via **Azure Key Vault** — a chave de acesso é lida em runtime e nunca fica exposta no código (boa prática de segurança).

In [2]:
# ============================================================
# Configuração de acesso ao ADLS Gen2 via Azure Key Vault
# A chave é lida do Key Vault em runtime — nunca fica exposta no código
# ============================================================
from notebookutils import mssparkutils
from pyspark.sql.functions import col
import time
from datetime import datetime

CONTA_STORAGE = "stalfalfabetizacao"

# Busca a chave de acesso direto do Key Vault
CHAVE_ACESSO = mssparkutils.credentials.getSecret(
    "https://kv-alfabetizacao.vault.azure.net/",
    "storage-account-key"
)

# Configura o Spark para usar a chave
spark.conf.set(
    f"fs.azure.account.key.{CONTA_STORAGE}.dfs.core.windows.net",
    CHAVE_ACESSO
)

# Caminhos base de cada camada
CAMINHO_BRONZE = f"abfss://bronze@{CONTA_STORAGE}.dfs.core.windows.net/ts_aluno/"
CAMINHO_SILVER = f"abfss://silver@{CONTA_STORAGE}.dfs.core.windows.net/ts_aluno/"
CAMINHO_GOLD   = f"abfss://gold@{CONTA_STORAGE}.dfs.core.windows.net/ts_aluno/"


print("Conexão configurada com sucesso!")
print(f"  Bronze : {CAMINHO_BRONZE}")
print(f"  Silver : {CAMINHO_SILVER}")
print(f"  Gold   : {CAMINHO_GOLD}")

StatementMeta(sparkpool, 31, 3, Finished, Available, Finished, False)

Conexão configurada com sucesso!
  Bronze : abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/ts_aluno/
  Silver : abfss://silver@stalfalfabetizacao.dfs.core.windows.net/ts_aluno/
  Gold   : abfss://gold@stalfalfabetizacao.dfs.core.windows.net/ts_aluno/


## 3. Monitoramento

O código monitora a qualidade e a performance dos datasets da camada Silver, verificando falhas de ingestão, valores nulos, duplicidades, relacionamentos entre tabelas e latência de processamento, além de registrar o tempo total de execução do pipeline.

In [9]:

# ============================================================
# FUNÇÃO ALERTA
# ============================================================

def enviar_alerta(mensagem):

    print(f"🚨 ALERTA: {mensagem}")

    logging.error(
        f"ALERTA DISPARADO: {mensagem}"
    )

# ============================================================
# INÍCIO PIPELINE
# ============================================================

inicio_pipeline = time.time()

logging.info(
    "Iniciando monitoramento da camada Prata"
)

nome = "TS_ALUNO"

# ============================================================
# LEITURA DATASET
# ============================================================

try:

    inicio_dataset = time.time()

    df = spark.read.parquet(
        CAMINHO_SILVER
    )

    volume = df.count()

    print(
        f"{nome}: {volume} registros encontrados."
    )

    logging.info(
        f"{nome}: {volume} registros encontrados."
    )

except Exception as erro:

    mensagem = (
        f"Falha na leitura do dataset "
        f"{nome}: {erro}"
    )

    logging.error(mensagem)

    enviar_alerta(mensagem)

    raise

# ============================================================
# MONITORAMENTO DE NULOS
# ============================================================

total_nulos = 0

COLUNAS_CRITICAS = [
    "NU_ANO_AVALIACAO",
    "ID_ALUNO",
    "ID_ESCOLA",
    "CO_UF",
    "CO_MUNICIPIO"
]

for coluna in COLUNAS_CRITICAS:

    qtd_nulos = df.filter(
        col(coluna).isNull()
    ).count()

    total_nulos += qtd_nulos

    if qtd_nulos > 0:

        print(
            f"ALERTA - {nome}: "
            f"coluna '{coluna}' possui "
            f"{qtd_nulos} nulos"
        )

        logging.warning(
            f"{nome}: coluna '{coluna}' "
            f"possui {qtd_nulos} nulos"
        )

if total_nulos == 0:

    print(
        f"{nome}: sem valores nulos"
    )

    logging.info(
        f"{nome}: sem valores nulos"
    )

# ============================================================
# MONITORAMENTO DE DUPLICIDADE
# ============================================================
#
# Ajuste as chaves conforme o dataset
#

CHAVES_NEGOCIO = [
    "NU_ANO_AVALIACAO",
    "ID_ALUNO",
    "ID_ESCOLA"
]

duplicados = (

    df

    .groupBy(
        CHAVES_NEGOCIO
    )

    .count()

    .filter(
        col("count") > 1
    )

    .count()

)

if duplicados > 0:

    mensagem = (
        f"{nome}: "
        f"{duplicados} registros duplicados"
    )

    print(mensagem)

    logging.warning(mensagem)

    enviar_alerta(mensagem)

else:

    print(
        f"{nome}: sem duplicidade"
    )

    logging.info(
        f"{nome}: sem duplicidade"
    )

# ============================================================
# ALERTA DE VOLUME ANORMAL
# ============================================================

if volume == 0:

    mensagem = (
        f"{nome}: dataset vazio"
    )

    enviar_alerta(mensagem)

else:

    print(
        f"{nome}: dataset contém dados"
    )

    logging.info(
        f"{nome}: dataset contém dados"
    )

# ============================================================
# LATÊNCIA DATASET
# ============================================================

fim_dataset = time.time()

latencia_dataset = (
    fim_dataset - inicio_dataset
)

print(
    f"{nome}: latência "
    f"{latencia_dataset:.2f}s"
)

logging.info(
    f"{nome}: latência "
    f"{latencia_dataset:.2f}s"
)

# ============================================================
# LATÊNCIA TOTAL PIPELINE
# ============================================================

fim_pipeline = time.time()

latencia_total = (
    fim_pipeline - inicio_pipeline
)

print(
    f"Tempo total do pipeline: "
    f"{latencia_total:.2f}s"
)

logging.info(
    f"Tempo total do pipeline: "
    f"{latencia_total:.2f}s"
)

print(
    "Monitoramento finalizado com sucesso."
)

logging.info(
    "Monitoramento finalizado com sucesso."
)

# ============================================================
# MONITORAMENTO DE RELACIONAMENTO
# TS_ALUNO x MUNICIPIO
# ============================================================

try:

    df_municipio = spark.read.parquet(
        f"abfss://silver@{CONTA_STORAGE}.dfs.core.windows.net/municipio/"
    )

    municipios_sem_relacionamento = (

        df

        .select("CO_MUNICIPIO")

        .distinct()

        .join(
            df_municipio
            .select("id_municipio")
            .distinct(),

            col("CO_MUNICIPIO") == col("id_municipio"),

            "left_anti"
        )

        .count()

    )

    if municipios_sem_relacionamento > 0:

        mensagem = (
            f"ALERTA - {nome}: "
            f"{municipios_sem_relacionamento} municípios "
            f"sem correspondência na tabela MUNICIPIO"
        )

        print(mensagem)

    else:

        print(
            f"{nome}: relacionamento MUNICIPIO válido"
        )

        logging.info(
            f"{nome}: relacionamento MUNICIPIO válido"
        )

except Exception as erro:

    mensagem = (
        f"Erro no monitoramento de relacionamento: "
        f"{erro}"
    )

    print(mensagem)

    logging.error(mensagem)

StatementMeta(sparkpool, 31, 10, Finished, Available, Finished, False)

TS_ALUNO: 2222164 registros encontrados.
TS_ALUNO: sem valores nulos
TS_ALUNO: sem duplicidade
TS_ALUNO: dataset contém dados
TS_ALUNO: latência 7.33s
Tempo total do pipeline: 7.33s
Monitoramento finalizado com sucesso.
ALERTA - TS_ALUNO: 17 municípios sem correspondência na tabela MUNICIPIO
